# Linear regression lab: what makes a medical bill bigger?
**MBI806B · Business Data Analytics with AI and ML**

This is the finished notebook for the lab. Use it to check your own work if you get stuck, not instead of doing it. Your report needs screenshots of *your* notebook, with your name in the title.

**Before you run anything:** click the folder icon on the left, then the upload button, and upload `insurance.csv` from Kaggle. The file disappears when Colab disconnects, so upload it again if you come back later.

## Step 1: Load the data
`pandas` is the Python tool for tables. We give it a short nickname, `pd`, because we'll type it a lot.

In [ ]:
import pandas as pd

df = pd.read_csv("insurance.csv")
df.head()

## Step 2: Get to know it
How big is it? What type is each column? What do the numbers look like?

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe().round(1)

In [ ]:
df["smoker"].value_counts()

## Step 3: Clean it
Three checks: anything missing, anything doubled up, anything a model can't read.

In [ ]:
df.isnull().sum()

In [ ]:
print(df.duplicated().sum())

In [ ]:
df[df.duplicated(keep=False)]

In [ ]:
df = df.drop_duplicates()
df.shape

A model only understands numbers. So `yes` becomes 1 and `no` becomes 0.

In [ ]:
df["smoker"] = df["smoker"].map({"yes": 1, "no": 0})
df.head()

## Step 4: Filter it
Pull out just the rows you want, then compare.

In [ ]:
smokers = df[df["smoker"] == 1]
non_smokers = df[df["smoker"] == 0]

print(len(smokers), "smokers")
print(len(non_smokers), "non-smokers")

In [ ]:
df.groupby("smoker")["charges"].mean().round()

## Step 5: Look before you model
One dot per person. Age along the bottom, their bill up the side.

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df["age"], df["charges"])
plt.xlabel("Age")
plt.ylabel("Charges ($)")
plt.show()

Three bands. Colour the dots by smoker and see what they are.

In [ ]:
plt.scatter(df["age"], df["charges"], c=df["smoker"], cmap="coolwarm")
plt.xlabel("Age")
plt.ylabel("Charges ($)")
plt.show()

## Step 6: Split, then train
Keep 20% of the people hidden from the model. We test it on them at the end, the way an exam uses questions you haven't seen.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[["age", "bmi", "children", "smoker"]]
y = df["charges"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(len(X_train), "people to learn from")
print(len(X_test), "people kept back for the test")

First, a model that only knows each person's age.

In [ ]:
from sklearn.linear_model import LinearRegression

model_age = LinearRegression()
model_age.fit(X_train[["age"]], y_train)

print("Score:", round(model_age.score(X_test[["age"]], y_test), 2))

Now give it all four columns.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Score:", round(model.score(X_test, y_test), 2))

## Step 7: Read what it learned

In [ ]:
print("Starting point:", round(model.intercept_))
pd.Series(model.coef_, index=X.columns).round()

In [ ]:
from sklearn.metrics import mean_absolute_error

predictions = model.predict(X_test)
print("Average miss in dollars:", round(mean_absolute_error(y_test, predictions)))

In [ ]:
plt.scatter(y_test, predictions)
plt.plot([0, 60000], [0, 60000], color="black")
plt.xlabel("Real charges ($)")
plt.ylabel("Predicted charges ($)")
plt.show()

## Step 8: Predict someone new
Made-up people only. Never put your own health details, or anybody else's, into a notebook.

In [ ]:
new_people = pd.DataFrame({
    "age":      [30, 30, 55],
    "bmi":      [25, 25, 32],
    "children": [0, 0, 2],
    "smoker":   [0, 1, 0],
})

model.predict(new_people).round()

## Your turn
Change the three people above to your own three made-up customers, run the cell again, and put the results in your report.